# Run a Distributed PyTorch TrainJob

This notebook uses [Kubeflow Trainer](https://www.kubeflow.org/docs/components/trainer/) to run a distributed [PyTorch DDP](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html) `TrainJob` against the EKS cluster. We train a simple Convolutional Neural Network (CNN) on the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset.

It replaces the previous Training Operator v1 `PyTorchJob` example: the per-framework `kubeflow.org` CRDs are gone, and jobs are now submitted as `TrainJob` resources (`trainer.kubeflow.org`) against a `torch-distributed` `ClusterTrainingRuntime`.

## Setup

Install the Kubeflow SDK, which talks to the Kubeflow Trainer APIs.

In [ ]:
!pip install -U kubeflow

Install PyTorch and Torchvision so the training function can also be run locally in the notebook.

In [ ]:
!pip install torch==2.10.0 torchvision==0.22.1

## Install AWS CLI

Used by the local kubeconfig, which is likely configured to use an exec plugin that calls `aws eks get-token` to obtain authentication tokens.

In [ ]:
!curl "https://awscli.amazonaws.com/awscli-exe-linux-aarch64.zip" -o ~/awscliv2.zip

In [ ]:
%%capture
!unzip ~/awscliv2.zip -d ~/

In [ ]:
!sudo ~/aws/install

In [ ]:
!aws --version

## Install Kubectl

In [ ]:
!curl -L "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/arm64/kubectl" -o ~/kubectl

In [ ]:
!sudo install -o root -g root -m 0755 ~/kubectl /usr/local/bin/kubectl

Now, let's validate the installation.

In [ ]:
!kubectl version --client

## Define the Training Function

A PyTorch DDP training function that:

* Downloads the Fashion MNIST dataset.
* Builds a simple convolutional neural network.
* Initializes the PyTorch distributed process group (Trainer injects `RANK`/`WORLD_SIZE`/`LOCAL_RANK`).
* Runs a sharded training loop.

The function takes plain keyword arguments — Kubeflow Trainer passes `CustomTrainer.func_args` to it as kwargs.

In [ ]:
def train_fashion_mnist(num_epochs=2, batch_size=100, lr=0.1, momentum=0.9):
    import os

    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler
    from torchvision import datasets, transforms

    # Define the PyTorch CNN model to be trained.
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(1, 20, 5, 1)
            self.conv2 = nn.Conv2d(20, 50, 5, 1)
            self.fc1 = nn.Linear(4 * 4 * 50, 500)
            self.fc2 = nn.Linear(500, 10)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            x = F.max_pool2d(x, 2, 2)
            x = F.relu(self.conv2(x))
            x = F.max_pool2d(x, 2, 2)
            x = x.view(-1, 4 * 4 * 50)
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return F.log_softmax(x, dim=1)

    # Use NCCL if a GPU is available, otherwise use Gloo as communication backend.
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    # Kubeflow Trainer sets RANK / WORLD_SIZE / LOCAL_RANK / MASTER_ADDR / MASTER_PORT
    # in each node's environment, so init_process_group needs no explicit args.
    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    # Create the model and load it onto the device.
    device = torch.device(f"{device}:{local_rank}")
    model = Net().to(device)
    model = nn.parallel.DistributedDataParallel(model)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    # Download the dataset only on the local rank 0 process, then fan out.
    if local_rank == 0:
        datasets.FashionMNIST("./data", train=True, download=True)
    dist.barrier()
    dataset = datasets.FashionMNIST(
        "./data",
        train=True,
        download=False,
        transform=transforms.Compose([transforms.ToTensor()]),
    )

    # Every worker gets a distributed shard of the dataset.
    train_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=DistributedSampler(dataset),
    )

    dist.barrier()
    for epoch in range(1, num_epochs + 1):
        model.train()
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = F.nll_loss(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if batch_idx % 10 == 0 and dist.get_rank() == 0:
                print(
                    "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                        epoch,
                        batch_idx * len(inputs),
                        len(train_loader.dataset),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )

    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")
    dist.destroy_process_group()

## Run the Training Locally

Submit the training function to the local Trainer backend to run it in an isolated subprocess before scaling out to the cluster.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

local_client = TrainerClient(backend_config=LocalProcessBackendConfig(cleanup_venv=True))

local_job = local_client.train(
    trainer=CustomTrainer(
        func=train_fashion_mnist,
        func_args={"num_epochs": 1},
        num_nodes=1,
    ),
)
for logline in local_client.get_job_logs(local_job, follow=True):
    print(logline)

## Create & Submit the Distributed TrainJob

We use the Kubeflow SDK `TrainerClient` to submit a `TrainJob` against the cluster. The job runs on the built-in `torch-distributed` `ClusterTrainingRuntime` (enabled in this cluster via the trainer chart's `runtimes.defaultEnabled=true`).

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

client = TrainerClient()

List the available Training Runtimes and pick `torch-distributed`.

In [ ]:
torch_runtime = None
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

assert torch_runtime is not None, "torch-distributed runtime not found"

Submit the distributed `TrainJob`. Kubeflow Trainer will train the model across 2 PyTorch nodes.

In [ ]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_fashion_mnist,
        func_args={"num_epochs": 6, "batch_size": 64, "lr": 0.01, "momentum": 0.9},
        # How many PyTorch nodes to use for distributed training.
        num_nodes=2,
        # Resources for each PyTorch node.
        resources_per_node={
            "cpu": 3,
            "memory": "16Gi",
            # Uncomment to schedule the TrainJob on GPU nodes.
            # "nvidia.com/gpu": 1,
        },
    ),
    runtime=torch_runtime,
)
print(job_name)

## Check the TrainJob Status

Using the Python SDK. The first TrainJob on a fresh node can take several minutes to reach `Running` while the (large CUDA) PyTorch runtime image is pulled, so we pass a generous `timeout`.

In [ ]:
client.wait_for_job_status(name=job_name, status={"Running"}, timeout=1800)
for step in client.get_job(name=job_name).steps:
    print(f"Step: {step.name}, Status: {step.status}, Devices: {step.device} x {step.device_count}")

Using Kubectl.

In [ ]:
!kubectl get trainjob {job_name}

## Get Pod Names

Using Kubectl. TrainJob pods are orchestrated via JobSet, so select on the JobSet label.

In [ ]:
!kubectl get pods -l jobset.sigs.k8s.io/jobset-name={job_name}

## TrainJob Training Logs

Using the Python SDK.

In [ ]:
for logline in client.get_job_logs(job_name, follow=True):
    print(logline)

Using Kubectl.

In [ ]:
!kubectl logs -f "$(kubectl get pods -l jobset.sigs.k8s.io/jobset-name={job_name},jobset.sigs.k8s.io/job-index=0 -o name | head -n1)"

## Verify TrainJob Completion

Wait for the TrainJob to reach a terminal status.

In [ ]:
client.wait_for_job_status(name=job_name, status={"Complete"}, timeout=1800)

## Delete the TrainJob

Using the Python SDK.

In [ ]:
client.delete_job(job_name)

Using Kubectl.

In [ ]:
!kubectl delete trainjob {job_name}